In [1]:

from pytorch_pretrained_bert.modeling import BertForSequenceClassification, BertConfig
import torch
from pytorch_pretrained_bert.tokenization import BertTokenizer
import sys
sys.path.append("/home/sdanisetty/projects/advpur/repos/TextFooler/BERT/")
from run_classifier import convert_examples_to_features
from run_classifier import AGProcessor
from torch.utils.data import (DataLoader, RandomSampler, SequentialSampler,
                              TensorDataset)
from tqdm import tqdm
import numpy as np
def accuracy(out, labels):
    outputs = np.argmax(out, axis=1)
    return np.sum(outputs == labels)

In [2]:
output_config_file = "/home/sdanisetty/projects/advpur/repos/ag/bert_config.json"  # Replace with your config file path
output_model_file = "/home/sdanisetty/projects/advpur/repos/ag/pytorch_model.bin"  # Replace with your model file path
config = BertConfig(output_config_file)
model = BertForSequenceClassification(config, num_labels=4)
model.load_state_dict(torch.load(output_model_file))

<All keys matched successfully>

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

04/22/2025 16:31:50 - INFO - pytorch_pretrained_bert.tokenization -   loading vocabulary file https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt from cache at /home/sdanisetty/.pytorch_pretrained_bert/26bc1ad6c0ac742e9b52263248f6d0f00068293b33709fae12320c0e35ccfbbb.542ce4285a40d23a559526243235df47c5f75c197f04f37d1a0c124c32c9a084


In [ ]:
# Input text for inference
input_text = "This is a sample input text."

# Tokenize the input text
# inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True)
# 
tokens = tokenizer.tokenize(input_text)
input_ids = tokenizer.convert_tokens_to_ids(tokens)
# Run inference
# with torch.no_grad():
#     outputs = model(**inputs)

# # Get predictions
# logits = outputs.logits
# predicted_class = torch.argmax(logits, dim=1).item()

# print(f"Predicted class: {predicted_class}")

In [ ]:
input_ids

[2023, 2003, 1037, 7099, 7953, 3793, 1012]

In [3]:
import sys
sys.path.append("/home/sdanisetty/projects/advpur/repos/TextFooler/BERT/")
from run_classifier import convert_examples_to_features
from run_classifier import AGProcessor
# train_features = convert_examples_to_features(train_examples, label_list, 128, tokenizer)

In [4]:
processor = AGProcessor()
eval_examples = processor.get_dev_examples('/home/sdanisetty/projects/advpur/data/ag/')
label_list = processor.get_labels()
eval_features = convert_examples_to_features(
            eval_examples, label_list, 128, tokenizer)

04/22/2025 15:35:32 - INFO - run_classifier -   *** Example ***
04/22/2025 15:35:32 - INFO - run_classifier -   guid: dev-0
04/22/2025 15:35:32 - INFO - run_classifier -   tokens: [CLS] fears for t n pension after talks . unions representing workers at turner new ##all say they are ' disappointed ' after talks with stricken parent firm federal mo ##gul . [SEP]
04/22/2025 15:35:32 - INFO - run_classifier -   input_ids: 101 10069 2005 1056 1050 11550 2044 7566 1012 9209 5052 3667 2012 6769 2047 8095 2360 2027 2024 1005 9364 1005 2044 7566 2007 16654 6687 3813 2976 9587 24848 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
04/22/2025 15:35:32 - INFO - run_classifier -   input_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [ ]:
all_input_ids = torch.tensor([f.input_ids for f in eval_features], dtype=torch.long)
all_input_mask = torch.tensor([f.input_mask for f in eval_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.segment_ids for f in eval_features], dtype=torch.long)
all_label_ids = torch.tensor([f.label_id for f in eval_features], dtype=torch.long)
eval_data = TensorDataset(all_input_ids, all_input_mask, all_segment_ids, all_label_ids)
eval_sampler = SequentialSampler(eval_data)
eval_dataloader = DataLoader(eval_data, sampler=eval_sampler, batch_size=16)

In [ ]:
model.eval().cuda()
eval_loss, eval_accuracy = 0, 0
nb_eval_steps, nb_eval_examples = 0, 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for input_ids, input_mask, segment_ids, label_ids in tqdm(eval_dataloader, desc="Evaluating"):
    input_ids = input_ids.to(device)
    input_mask = input_mask.to(device)
    segment_ids = segment_ids.to(device)
    label_ids = label_ids.to(device)

    with torch.no_grad():
        tmp_eval_loss = model(input_ids, segment_ids, input_mask, label_ids)
        logits = model(input_ids, segment_ids, input_mask)

    logits = logits.detach().cpu().numpy()
    label_ids = label_ids.to('cpu').numpy()
    tmp_eval_accuracy = accuracy(logits, label_ids)

    eval_loss += tmp_eval_loss.mean().item()
    eval_accuracy += tmp_eval_accuracy

    nb_eval_examples += input_ids.size(0)
    nb_eval_steps += 1

Evaluating:   0%|          | 0/475 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 475/475 [01:09<00:00,  6.87it/s]


In [ ]:
eval_accuracy / nb_eval_examples

0.9423684210526316

In [12]:
all_input_ids[0].shape, all_segment_ids[0].shape, all_input_mask[0].shape

(torch.Size([128]), torch.Size([128]), torch.Size([128]))

In [13]:
all_input_ids[0]

tensor([  101, 10069,  2005,  1056,  1050, 11550,  2044,  7566,  1012,  9209,
         5052,  3667,  2012,  6769,  2047,  8095,  2360,  2027,  2024,  1005,
         9364,  1005,  2044,  7566,  2007, 16654,  6687,  3813,  2976,  9587,
        24848,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [14]:
all_segment_ids[0]

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [35]:
all_input_mask[0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
model.cuda()
# og_text = "E-mail scam targets police chief Wiltshire Police warns about “phishing” after its fraud squad chief was targeted."
# adv_text = "E-mail scam targets gendarmerie chief Wiltshire Police warns about “phishing” after its deception battalion massa was targeted."
# # text = "E-mail scam scam police chief Wortshire Police warns about “phishing�; after a hoax battalion leiter was targeted."
# text = "E-mail scam targets gammerie chiefrotshire Police warns about “phishing” after its deception battalion massa being targeted."
og_text = "aussie ruling party leads in election polls , but gap narrows the government of prime minister john howard had a narrow lead in opinion polls heading into the final week of campaigning ahead of the australian federal election , but the opposition labor party was narrowing the gap , according"
adv_text = "aussie ruling party leads in election polls , but gap narrows the gov of prime minister john howard had a ltd culminate in opinion polls heading into the final week of campaigning ahead of the aus federal election , but the opposition labor party was downsize the disparity , according"
text = "a the ruling party leads from<|endoftext|>os, but gap narrows the gov of prime minister john Chard had a ltd culminate of opinion polls heading into the final week of campaigning ahead of theusus federal election, while the opposition labor party was down to the disparity, according"
class_names = {0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech"}
max_seq_length = 128
# for t in [og_text, adv_text, text]:
#     tokens = tokenizer.tokenize(t)
#     tokens = ["[CLS]"] + tokens + ["[SEP]"]
#     max_seq_length = 128
#     input_ids = tokenizer.convert_tokens_to_ids(tokens)
#     segment_ids = [0] * len(input_ids)
#     input_mask = [1] * len(input_ids)
#     padding = [0] * (max_seq_length - len(input_ids))
#     input_ids += padding
#     input_mask += padding
#     segment_ids += padding
#     out = model(torch.tensor([input_ids], device='cuda'), torch.tensor([segment_ids], device='cuda'), torch.tensor([input_mask], device='cuda'))
#     # print(out)
#     print(class_names[np.argmax(out.detach().cpu().numpy(), axis=1)[0]])
tokens = tokenizer.tokenize(adv_text)
tokens = ["[CLS]"] + tokens + ["[SEP]"]
input_ids = tokenizer.convert_tokens_to_ids(tokens)
segment_ids = [0] * len(input_ids)
input_mask = [1] * len(input_ids)
padding = [0] * (max_seq_length - len(input_ids))
input_ids += padding
input_mask += padding
segment_ids += padding
out = model(torch.tensor([input_ids], device='cuda'), torch.tensor([segment_ids], device='cuda'), torch.tensor([input_mask], device='cuda'))
print(out)
class_names[np.argmax(out.detach().cpu().numpy(), axis=1)[0]]

tensor([[ 1.3113, -1.2998,  3.1289, -4.7444]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


'Business'

In [ ]:
# out = model(torch.tensor([input_ids], device='cuda'), torch.tensor([segment_ids], device='cuda'), torch.tensor([input_mask], device='cuda'))
# print(out)
# class_names = {0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech"}
# class_names[np.argmax(out.detach().cpu().numpy(), axis=1)[0]]

tensor([[ 2.5158, -2.4636,  3.2696, -4.1575]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


'Business'